In [1]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
import pyspark.pandas as ps

import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [2]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole, AdditionalStatisticRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer

In [6]:
# --- 1. Настройки окружения для macOS (Важно!) ---
# На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных (как у вас было)
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация Кластера ---
# Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048 

MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

print(f"🚀 Запуск в режиме: {MASTER_URL}")

sp_s = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Память драйвера (остается у вас)
    .config("spark.driver.memory", "2g") 
    # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "4")
    .config("spark.executor.instances", NUM_EXECUTORS)
    # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
    .config("spark.memory.fraction", "0.6")
    .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
    .getOrCreate()
)

sp_s.sparkContext.setLogLevel("WARN")

# --- 3. Проверка конфигурации ---
print(f"✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# Проверка количества экзекуторов (может занять пару секунд на старт)
import time
time.sleep(3) 
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 4. Тест на распределение (Пример) ---
# Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
def print_executor_info(iterator):
    import os
    # Получаем ID экзекутора из переменных окружения процесса
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

# Создаем датафрейм и применяем трансформацию
df = sp_s.range(0, 10, 1, 4) # 4 партиции
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# sp_s.stop() 

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/06/17 16:37:11 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.Con

✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2g


📊 Активных экзекуторов (проверка через RDD): 2



🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 566616
Executor ID: Driver/Local, PID: 566610
Executor ID: Driver/Local, PID: 566672
Executor ID: Driver/Local, PID: 566665


In [3]:
n = 5000  # увеличьте для теста IVF-индексов
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [ ]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 5000, nn),
        '1': np.random.randint(0, 5000, nn),
        '2': np.random.randint(0, 5000, nn),
        '3': np.random.randint(0, 5000, nn),
        '4': np.random.randint(0, 5000, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

# index_df = pd.concat([index_df, pd.DataFrame(data={
#     '0': np.nan,
#     '1': np.nan,
#     '2': np.nan,
#     '3': np.nan,
#     '4': np.nan,
#     'group': np.nan
# }, index=[0])]).reset_index(drop=True)
# index_df

In [ ]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

In [ ]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session
    # session=sp_s
)

index_ds

In [ ]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        OnRoleExperiment(
            executors=[
                TTest(
                    grouping_role=TreatmentRole(),
                    compare_by="matched_pairs",
                    baseline_role=AdditionalMatchingRole(),
                )
            ],
            role=FeatureRole()
        )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

DummyEncoder
executor.key = DummyEncoder┴┴┆feat_cat_C┆; dt = 0.0112c
MahalanobisDistance
executor.key = MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||┆feat_cat_B┆', 'DummyEncoder||┆feat_cat_C┆']; dt = 0.0285c
TypeCaster
executor.key = TypeCaster┴┴; dt = 0.0084c
FaissNearestNeighbors
executor.key = FaissNearestNeighbors┴┴; dt = 0.5682c
Bias
executor.key = Bias┴┴['target', 'target_matched']; dt = 0.1165c
MatchingMetrics
executor.key = MatchingMetrics┴┴['target', 'target_matched']; dt = 0.2546c
MatchingAnalyzer
executor.key = MatchingAnalyzer┴┴; dt = 0.0029c
OnRoleExperiment
GroupTTest
executor.key = TTest┴┴; dt = 0.0626c
GroupTTest
executor.key = TTest┴┴; dt = 0.0513c
GroupTTest
executor.key = TTest┴┴; dt = 0.0087c
executor.key = OnRoleExperiment┴┴; dt = 0.1327c


/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:239: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(
/home/eric/HypEx/HypEx/hypex/comparators/abstract.py:239: UserWarning: baseline_field_data must have only one column when the comparison is done by matched_pairs. 2 passed. FaissNearestNeighbors┴┴┴0 will be used.
  warnings.warn(


In [7]:
pandas_result.analysis_tables['GroupTTest┴┴feat_num_1']

,"(0,)┆feat_num_1","(1,)┆feat_num_1"
0,"[[[ 0\np-value 0.712607\n\n1 rows × 1 columns], [ 0\nstatistic -0.368375\n\n1 rows × 1 columns], [ 0\npass 0.0\n\n1 rows × 1 columns]]]","[[[ 0\np-value 0.662309\n\n1 rows × 1 columns], [ 0\nstatistic 0.43676\n\n1 rows × 1 columns], [ 0\npass 0.0\n\n1 rows × 1 columns]]]"


In [ ]:
pandas_result.field_search(AdditionalStatisticRole())

In [ ]:
pandas_result.analysis_tables

In [8]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
spark_result = spark_experiment.execute(ExperimentData(dataset))

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


DummyEncoder


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


executor.key = DummyEncoder┴┴┆feat_cat_C┆; dt = 1.5297c
MahalanobisDistance


26/06/17 16:37:49 WARN AttachDistributedSequenceExec: clean up cached RDD(49) in AttachDistributedSequenceExec(154)
26/06/17 16:37:50 WARN AttachDistributedSequenceExec: clean up cached RDD(65) in AttachDistributedSequenceExec(359)
26/06/17 16:37:51 WARN AttachDistributedSequenceExec: clean up cached RDD(79) in AttachDistributedSequenceExec(716)
26/06/17 16:37:51 WARN AttachDistributedSequenceExec: clean up cached RDD(87) in AttachDistributedSequenceExec(740)
26/06/17 16:37:53 WARN AttachDistributedSequenceExec: clean up cached RDD(116) in AttachDistributedSequenceExec(1640)
26/06/17 16:37:53 WARN AttachDistributedSequenceExec: clean up cached RDD(124) in AttachDistributedSequenceExec(1664)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26

...

     feat_num_1 feat_num_2
626    4.796678  -3.237022
625     1.28831  -1.942176
632    4.737229   1.980413
629    2.845365   0.007704
630   -1.648725  -3.980413
...         ...        ...
3038   0.083147   0.009523
3039  -0.047309  -0.084822
3040   0.067148   0.045113
3041   0.010883   -0.01312
3042  -0.056052   0.018952

3043 rows × 2 columns


/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/06/17 16:39:28 WARN AttachDistributedSequenceExec: clean up cached RDD(3628) in AttachDistributedSequenceExec(96833)
26/06/17 16:39:28 WARN AttachDistributedSequenceExec: clean up cached RDD(3636) in AttachDistributedSequenceExec(96860)
26/06/17 16:39:28 WARN AttachDistributedSequenceExec: clean up cached RDD(3642) in AttachDistributedSequenceExec(96909)
26/06/17 16:39:29 WARN AttachDistributedSequenceExec: clean up cached RDD(3648) in AttachDis

executor.key = Bias┴┴['target', 'target_matched']; dt = 64.5272c
MatchingMetrics


26/06/17 16:39:33 WARN AttachDistributedSequenceExec: clean up cached RDD(3897) in AttachDistributedSequenceExec(108297)
26/06/17 16:39:33 WARN AttachDistributedSequenceExec: clean up cached RDD(3905) in AttachDistributedSequenceExec(108321)
26/06/17 16:39:33 WARN AttachDistributedSequenceExec: clean up cached RDD(3911) in AttachDistributedSequenceExec(108364)
26/06/17 16:39:33 WARN AttachDistributedSequenceExec: clean up cached RDD(3917) in AttachDistributedSequenceExec(108444)
26/06/17 16:39:33 WARN AttachDistributedSequenceExec: clean up cached RDD(3923) in AttachDistributedSequenceExec(108566)
26/06/17 16:39:38 WARN AttachDistributedSequenceExec: clean up cached RDD(4229) in AttachDistributedSequenceExec(125338)
26/06/17 16:39:39 WARN AttachDistributedSequenceExec: clean up cached RDD(4237) in AttachDistributedSequenceExec(125362)
26/06/17 16:39:39 WARN AttachDistributedSequenceExec: clean up cached RDD(4243) in AttachDistributedSequenceExec(125405)
26/06/17 16:39:39 WARN AttachDis

executor.key = MatchingMetrics┴┴['target', 'target_matched']; dt = 318.8396c
MatchingAnalyzer
executor.key = MatchingAnalyzer┴┴; dt = 0.0027c


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [9]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT     0.016509        0.260494  0.949466 -0.494059  0.527078
 ATC     0.074782        0.618789  0.903808 -1.138045  1.287609
 ATE     0.051974        0.377098  0.890378 -0.687138  0.791086
 
 3 rows × 5 columns}

In [10]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()